# 🚀 Stage 4: QLoRA Fine-Tuning (Model M2: Vanilla SFT)
**Project:** Reduction Ladder for Code & Multi-Arm Mitigation  
**Organization:** Orange Innovation Labs  
**Authors:** Omar Abdelhamid, Nour Walid  
**Supervisor:** Dr. Ghada Soliman  

---

### 🎯 Objectives:
1. Fine-tune **Model M2 (Vanilla CoT SFT)** on `data/distillation/sft_positive_cot.jsonl`.
2. Base model: `Qwen/Qwen2.5-Coder-1.5B-Instruct` in 4-bit NF4 with gradient checkpointing.
3. Enforce strict VRAM budgeting: Peak VRAM ≈ 6.4 GB (well within the RTX 3070 8GB limit).
4. Save adapter weights to `checkpoints/qlora_vanilla_adapter`.

| Model | Training Method | Dataset | Output Checkpoint |
|---|---|---|---|
| **M2** | 4-bit NF4 QLoRA ($r=16, \alpha=32$) | `sft_positive_cot.jsonl` | `checkpoints/qlora_vanilla_adapter` |

> 📌 **Mitigation Arms Roadmap:**  
> - **Arm 1 (Inv-GRPO)**: Trained via `notebooks/arm_01_inv_grpo.ipynb` (Primary Innovation).
> - **Arm 2B (Contrastive DPO)**: Configured in `notebooks/arm_02_contrastive_sft.ipynb` for Phase 2.
> - **Arms 3 & 4 (AST-RL, Step-RLVR)**: Demonstrated in `notebooks/arm_03_ast_rl.ipynb` and `arm_04_step_rlvr.ipynb`.

In [1]:
import os
import sys
import torch

# Ensure project root is on sys.path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("."))

from src.core.config import get_settings
from src.stage4_training.qlora_finetune import QLoRAFineTuner

settings = get_settings()
os.makedirs(settings.storage.checkpoints_dir, exist_ok=True)

print("[OK] QLoRA Training Engine Environment:")
print(f"   CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Device     : {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   Total VRAM     : {total_vram:.2f} GB")
print(f"   Base Model     : {settings.models.student_model}")
print(f"   LoRA Rank (r)  : {settings.qlora.r}")
print(f"   LoRA Alpha     : {settings.qlora.alpha}")
print(f"   Batch Size     : {settings.qlora.batch_size} (Grad Accum: {settings.qlora.gradient_accumulation_steps})")
print(f"   Epochs         : {settings.qlora.epochs}")

c:\Users\Lenovo\anaconda3\envs\unsloth_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\Lenovo\anaconda3\envs\unsloth_env\lib\site-packages\transformers\utils\hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


[OK] QLoRA Training Engine Environment:
   CUDA Available : True
   GPU Device     : NVIDIA GeForce RTX 3070 Ti Laptop GPU
   Total VRAM     : 8.00 GB
   Base Model     : Qwen/Qwen2.5-Coder-1.5B-Instruct
   LoRA Rank (r)  : 16
   LoRA Alpha     : 32
   Batch Size     : 4 (Grad Accum: 4)
   Epochs         : 3


---
## 1. VRAM Status Check & Data Path Verification

In [2]:
QLoRAFineTuner.log_vram()

data_path = os.path.join(settings.storage.distillation_cache_dir, settings.distillation.vanilla_file)
adapter_output_dir = settings.qlora.vanilla_output_dir

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Training data not found at '{data_path}'. Please run 'nb_03_distillation.ipynb' first!")

print(f"[OK] Training data verified at: {data_path}")
print(f"[OK] Target checkpoint path : {adapter_output_dir}")

[VRAM] Used: 0.00 GB | Reserved: 0.00 GB | Total: 8.00 GB
[OK] Training data verified at: data/distillation\sft_positive_cot.jsonl
[OK] Target checkpoint path : checkpoints/qlora_vanilla_adapter


---
## 2. Train Model M2 (Vanilla CoT SFT)

Loads `Qwen2.5-Coder-1.5B-Instruct` in 4-bit NF4 with double quantization and trains for 3 epochs.

In [3]:
vanilla_tuner = QLoRAFineTuner(
    variant="vanilla",
    data_path=data_path,
    model_name=settings.models.student_model,
)

vanilla_adapter_path = vanilla_tuner.train(
    output_dir=adapter_output_dir
)

# Clean up memory after training
del vanilla_tuner
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n[OK] M2 (Vanilla SFT) training complete!")
print(f"   Adapter saved to: {vanilla_adapter_path}")
QLoRAFineTuner.log_vram()

[QLoRA] Loaded 500 training examples from 'data/distillation\sft_positive_cot.jsonl'


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[QLoRA] Loading tokenizer: 'Qwen/Qwen2.5-Coder-1.5B-Instruct'...
[QLoRA] Loading 4-bit NF4 model: 'Qwen/Qwen2.5-Coder-1.5B-Instruct'...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


...

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.



[QLoRA] ▶ Starting VANILLA SFT training...
[QLoRA]   Variant:    vanilla
[QLoRA]   Data:       data/distillation\sft_positive_cot.jsonl (500 samples)
[QLoRA]   Output:     checkpoints/qlora_vanilla_adapter
[QLoRA]   Epochs:     3
[QLoRA]   Eff. Batch: 16
[QLoRA]   LR:         0.0002



Step,Training Loss
10,1.177500
20,0.587400
30,0.510200
40,0.480000
50,0.482500
60,0.470700
70,0.428700
80,0.433700
90,0.414400


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 974a9f44-3514-4151-b4a1-9461f5c8796d)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 43bbeca4-305b-43ba-bf2a-d20fa7bb9187)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct/resolve/main/config.json
Retrying in 2s [Retry 2/5].


[QLoRA] ✅ Training complete. Saving LoRA adapter to 'checkpoints/qlora_vanilla_adapter'...
[QLoRA] ✅ Adapter saved successfully to 'checkpoints/qlora_vanilla_adapter'

[OK] M2 (Vanilla SFT) training complete!
   Adapter saved to: checkpoints/qlora_vanilla_adapter
[VRAM] Used: 1.30 GB | Reserved: 1.97 GB | Total: 8.00 GB


---
## 3. Checkpoint Verification

Ensures all required adapter weights (`adapter_model.safetensors`, `adapter_config.json`, and tokenizer files) are intact.

In [4]:
if os.path.exists(adapter_output_dir):
    files = os.listdir(adapter_output_dir)
    print(f"[OK] M2 Checkpoint Directory Found ({len(files)} files):")
    for f in sorted(files):
        size_kb = os.path.getsize(os.path.join(adapter_output_dir, f)) / 1024
        print(f"     - {f:30s} ({size_kb:.1f} KB)")
else:
    print(f"[PENDING] Checkpoint not yet generated at: {adapter_output_dir}")

print("\n-> Next Step: Run 'nb_05_post_training_eval.ipynb' to evaluate and compare M1 (Baseline) vs M2 (Vanilla SFT)!")

[OK] M2 Checkpoint Directory Found (12 files):
     - README.md                      (1.5 KB)
     - adapter_config.json            (1.1 KB)
     - adapter_model.safetensors      (72178.8 KB)
     - added_tokens.json              (0.6 KB)
     - checkpoint-32                  (4.0 KB)
     - checkpoint-64                  (4.0 KB)
     - checkpoint-93                  (4.0 KB)
     - merges.txt                     (1632.7 KB)
     - special_tokens_map.json        (0.6 KB)
     - tokenizer.json                 (11154.2 KB)
     - tokenizer_config.json          (7.4 KB)
     - vocab.json                     (2711.8 KB)

-> Next Step: Run 'nb_05_post_training_eval.ipynb' to evaluate and compare M1 (Baseline) vs M2 (Vanilla SFT)!
